# Solutions · 01 · Mole balances and the ideal reactors

Worked solutions to the exercises of [notebook 01](../notebooks/01_mole_balances_and_ideal_reactors.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engreact is missing): install it from GitHub
    import engreact
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engreact"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize
from engreact import ideal
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

A liquid-phase reaction with -r_A = k C_A^0.5 (k = 0.05 (mol/m3)^0.5/s, C_A0 = 400 mol/m3, F_A0 = 1 mol/s) is to reach 90 % conversion. Size a CSTR and a PFR, and explain why the ratio is smaller than for first order.

In [2]:
k, C_A0, F_A0 = 0.05, 400.0, 1.0
rate_half = lambda X: k * (C_A0 * (1 - X)) ** 0.5
for X in (0.5, 0.9):
    Vc, Vp = ideal.cstr_volume(F_A0, X, rate_half), ideal.pfr_volume(F_A0, X, rate_half)
    print(f"X = {X}: CSTR {Vc:.3f} m3, PFR {Vp:.3f} m3, ratio {Vc/Vp:.2f}")
rate_first = lambda X: 0.05 * 400.0 ** 0.5 / 400.0 * C_A0 * (1 - X)      # a first-order law with the same initial rate
print(f"for comparison, first order at X = 0.9: ratio {ideal.cstr_volume(F_A0, 0.9, rate_first)/ideal.pfr_volume(F_A0, 0.9, rate_first):.2f}")

X = 0.5: CSTR 0.707 m3, PFR 0.586 m3, ratio 1.21
X = 0.9: CSTR 2.846 m3, PFR 1.368 m3, ratio 2.08
for comparison, first order at X = 0.9: ratio 3.91


A half-order rate falls more slowly with conversion than a first-order one, so the CSTR's penalty for
running at the outlet concentration is smaller: the volume ratio at 90 % is about 2 against 3.9 for first
order. The lower the order, the less the reactor type matters; at zero order it does not matter at all.

## Exercise 2

For a zero-order reaction, show numerically that the CSTR and PFR volumes are equal for any conversion, and explain why with the Levenspiel picture.

In [3]:
rate0 = lambda X: 0.1 + 0 * X                                          # zero order: constant rate
for X in (0.3, 0.6, 0.9):
    print(f"X = {X}: CSTR {ideal.cstr_volume(1.0, X, rate0):.3f} m3, PFR {ideal.pfr_volume(1.0, X, rate0):.3f} m3")

X = 0.3: CSTR 3.000 m3, PFR 3.000 m3
X = 0.6: CSTR 6.000 m3, PFR 6.000 m3
X = 0.9: CSTR 9.000 m3, PFR 9.000 m3


With a constant rate the Levenspiel curve is flat, so the rectangle and the area under the curve coincide:
every mole is converted at the same rate wherever it is, and the reactor's mixing pattern is irrelevant.

## Exercise 3

**Implement it yourself:** write the CSTR balance for a first-order reaction as a residual function and solve it with brentq for X at tau = 20 min; compare with `ideal.cstr_conversion_first_order`.

In [4]:
k, tau = 0.05 / 60, 1200.0
X_hand = optimize.brentq(lambda X: X - k * tau * (1 - X), 0, 1)
print(f"by hand: X = {X_hand:.4f}; library: {ideal.cstr_conversion_first_order(k, tau):.4f}")

by hand: X = 0.5000; library: 0.5000


The CSTR balance $X = k\tau(1 - X)$ is linear here and hardly needs a root finder; the same three lines
solve it for any rate law where it is not.